In [17]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

In [18]:
# ---------------------------
# CUDA / cuDNN setup
# ---------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True  # enable cudnn autotuner for convs
# (Optional) slightly faster matmul kernels on Ampere+:
try:
    torch.set_float32_matmul_precision('medium')
except Exception:
    pass

print(device)

cuda


### Data 
Input is resized to the sensor's size of 126 X 126 


In [19]:
def make_loaders(data_root='./data', batch_size=256, workers=4):
    transform_train = T.Compose([
        T.Resize((126, 126), interpolation=T.InterpolationMode.BILINEAR, antialias=True),
        T.RandomHorizontalFlip(),
        T.ToTensor(),  # [0,1]
        # T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),  # Optional: zero-center
    ])
    transform_test = T.Compose([
        T.Resize((126, 126), interpolation=T.InterpolationMode.BILINEAR, antialias=True),
        T.ToTensor(),
        # T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
    ])

    trainset = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform_train)
    testset  = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform_test)

    # pin_memory+non_blocking enables async H2D copies
    train_loader = torch.utils.data.DataLoader(
        trainset, batch_size=batch_size, shuffle=True,
        num_workers=workers, pin_memory=True, persistent_workers=True
    )
    test_loader = torch.utils.data.DataLoader(
        testset, batch_size=batch_size, shuffle=False,
        num_workers=workers, pin_memory=True, persistent_workers=True
    )
    return train_loader, test_loader

In [20]:
# ---------------------------
# Binarization with STE
# ---------------------------
class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        out = x.new_empty(x.size())
        out.copy_(x)
        out[out >= 0] = 1.0
        out[out <  0] = -1.0
        return out

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        # Saturated STE: pass grad inside [-1,1], block outside
        grad_in = grad_out.clone()
        grad_in[x.abs() > 1.0] = 0.0
        return grad_in

binarize = BinarizeSTE.apply

class BinaryActivation(nn.Module):
    def forward(self, x):
        return binarize(x)

In [21]:
# ---------------------------
# Binary layers (weights binarized in forward)
# ---------------------------
class BinaryConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        # bias=False is typical; BN absorbs shift. But allow bias if passed.
        super().__init__(*args, **kwargs)

    def forward(self, x):
        # Binarize weights for forward (real weights kept for update)
        w_b = binarize(self.weight)
        return F.conv2d(input=x, weight=w_b, bias=self.bias, stride=self.stride, padding=self.padding, dilation=self.dilation, groups=self.groups)

class BinaryLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__(in_features, out_features, bias=bias)

    def forward(self, x):
        w_b = binarize(self.weight)
        return F.linear(x, w_b, self.bias)

### Model
Input is CIFAR10 dataset. This will later change to depth image

In [22]:


class BinaryCIFAR10Net(nn.Module):
    """
    Channel Dimension: {H+2P-(K-1)-1}/S + 1
    """
    def __init__(self, last_real_logits=True):
        super().__init__()

        # Block 1
        # conv : in_channel / out_channel / kernel_size / stride / padding 
        self.conv1 = BinaryConv2d(3, 32, 3, 3, 0, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.act1  = BinaryActivation()

        self.conv2 = BinaryConv2d(32, 85, 3, 3, 0, bias=False)
        self.bn2   = nn.BatchNorm2d(85)
        self.act2  = BinaryActivation()

        self.flatten = nn.Flatten()
        # Final logits (keep real weights for best accuracy)
        self.fc_out = (nn.Linear(85*14*14, 10, bias=True) if last_real_logits else BinaryLinear(85*14*14, 10, bias=True))

    def forward(self, x):
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.conv2(x)))

        x = self.flatten(x)
        x = self.fc_out(x)
        return x
    

In [23]:

# ---------------------------
# Train / Eval
# ---------------------------
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    crit = nn.CrossEntropyLoss()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = crit(logits, y)
        loss_sum += loss.item() * y.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total, loss_sum / total

def train(model, train_loader, test_loader, epochs=20, lr=1e-3, amp=True, ckpt_path='bnn_cifar10.pth'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    scaler = torch.cuda.amp.GradScaler(enabled=(amp and device.type == 'cuda'))

    best_acc = 0.0
    for ep in range(1, epochs+1):
        model.train()
        running = 0.0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(amp and device.type == 'cuda')):
                logits = model(x)
                loss = crit(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * y.size(0)

        scheduler.step()
        train_loss = running / len(train_loader.dataset)
        test_acc, test_loss = evaluate(model, test_loader, device)
        print(f"Epoch {ep:02d}/{epochs} | train_loss {train_loss:.4f} | "
              f"test_loss {test_loss:.4f} | test_acc {test_acc*100:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save({'model': model.state_dict(),
                        'acc': best_acc,
                        'epoch': ep}, ckpt_path)
            print(f"  ✔ Saved checkpoint: {ckpt_path} (acc={best_acc*100:.2f}%)")

    return best_acc


@torch.no_grad()
def predict_single(model, img_tensor_hw3):  # img: [H,W,3] or [3,H,W] with ToTensor (we expect [3,H,W])
    model.eval()
    if img_tensor_hw3.dim() == 3 and img_tensor_hw3.shape[0] != 3:
        img_tensor_hw3 = img_tensor_hw3.permute(2,0,1)
    x = img_tensor_hw3.unsqueeze(0).to(device)
    logits = model(x)
    pred = logits.argmax(1).item()
    return pred, logits.softmax(1).squeeze(0).cpu()

In [24]:
train_loader, test_loader = make_loaders(batch_size=256, workers=4)

model = BinaryCIFAR10Net(
    last_real_logits=True   # keep final FC real for accuracy
)

best_acc = train(model, train_loader, test_loader, epochs=300, lr=5e-4, amp=True)
print(f"Best test accuracy: {best_acc*100:.2f}%")

# Load best and test again
ckpt = torch.load('bnn_cifar10.pth', map_location='cpu')
model.load_state_dict(ckpt['model'])
model.to(device)
acc, _ = evaluate(model, test_loader, device)
print(f"Reloaded checkpoint accuracy: {acc*100:.2f}%")

/tmp/ipykernel_77785/552488571.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp and device.type == 'cuda'))
/tmp/ipykernel_77785/552488571.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(amp and device.type == 'cuda')):


Epoch 01/300 | train_loss 1.9698 | test_loss 2.1496 | test_acc 31.73%
  ✔ Saved checkpoint: bnn_cifar10.pth (acc=31.73%)
Epoch 02/300 | train_loss 1.9423 | test_loss 2.3042 | test_acc 31.63%
Epoch 03/300 | train_loss 1.9058 | test_loss 1.9749 | test_acc 37.54%
  ✔ Saved checkpoint: bnn_cifar10.pth (acc=37.54%)
Epoch 04/300 | train_loss 1.8748 | test_loss 2.0039 | test_acc 36.70%
Epoch 05/300 | train_loss 1.8293 | test_loss 2.0289 | test_acc 36.10%
Epoch 06/300 | train_loss 1.7847 | test_loss 2.2781 | test_acc 30.97%
Epoch 07/300 | train_loss 1.7445 | test_loss 2.0325 | test_acc 36.13%
Epoch 08/300 | train_loss 1.7199 | test_loss 2.0899 | test_acc 36.14%
Epoch 09/300 | train_loss 1.6635 | test_loss 2.1030 | test_acc 38.08%
  ✔ Saved checkpoint: bnn_cifar10.pth (acc=38.08%)
Epoch 10/300 | train_loss 1.6686 | test_loss 1.8726 | test_acc 41.32%
  ✔ Saved checkpoint: bnn_cifar10.pth (acc=41.32%)
Epoch 11/300 | train_loss 1.6387 | test_loss 1.9185 | test_acc 40.65%
Epoch 12/300 | train_loss 